In [1]:
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report


In [2]:
from pathlib import Path

# Root with split subfolders containing per-slide NPZs
NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_5x_norm2\unspecified")  # contains train/, val/, test/

# CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")

LABEL_MAP = {"FA": 0, "PT": 1}  # weak labels
RENORM_ROWS = True               # renormalize each patch prob row before averaging


skip to block below if files already in split dir

In [3]:
def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=False)
    qq = d["qq"]          # (N, K) patch x prototype probs
    coords = d["coords"]  # (N, 2) (unused for training, but could be kept)
    mask = d["mask"]      # (N,)
    return qq, coords, mask

def slide_vector_from_qq(qq, mask=None, renorm=True):
    """Aggregate patch probs → one K-dim feature per slide (mean qq)."""
    if mask is None:
        mask = np.ones((qq.shape[0],), dtype=bool)
    qq = qq[mask]
    if renorm:
        row_sums = qq.sum(axis=1, keepdims=True)
        qq = qq / (row_sums + 1e-12)
    return qq.mean(axis=0)  # shape (K,)

def parse_label_from_slide_id(slide_id: str):
    """Weak label from slide name: returns 1 for PT, 0 for FA."""
    sid = slide_id.strip().upper()
    if sid.startswith("PT"):
        return 1
    if sid.startswith("FA"):
        return 0
    raise ValueError(f"Cannot infer label from slide_id='{slide_id}'. Expect names starting with 'FA' or 'PT'.")


In [ ]:
import shutil, re, pandas as pd, numpy as np
from pathlib import Path
# # Root with split subfolders containing per-slide NPZs
# NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_10x_norm2\unspecified")  # contains train/, val/, test/

# # CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
# CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
# CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
# CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")

def canon(s: str) -> str:
    # Uppercase and strip spaces/underscores/hyphens for tolerant matching
    return re.sub(r"[\s_-]+", "", s.strip().upper())

# Index all existing NPZ by canonical stem
all_npz = list(NPZ_ROOT.rglob("*.npz"))
idx = {}
for p in all_npz:
    key = canon(p.stem)
    # keep first occurrence; warn on collisions
    idx.setdefault(key, p)

def load_ids(csv_path: Path, split: str):
    df = pd.read_csv(csv_path, dtype=str)
    assert {"slide_id","label"}.issubset(df.columns)
    df["slide_id"] = df["slide_id"].astype(str)
    return [(row.slide_id, split) for row in df.itertuples(index=False)]

targets = []
targets += load_ids(CSV_TRAIN, "train")
targets += load_ids(CSV_VAL,   "val")
targets += load_ids(CSV_TEST,  "test")

copied, missing = 0, []
for slide_id, split in targets:
    key = canon(slide_id)
    src = idx.get(key)
    if src is None:
        missing.append(slide_id)
        continue
    dst_dir = NPZ_ROOT / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{slide_id}.npz"  # keep original slide_id formatting in filename
    if not dst.exists():
        # shutil.copy2(src, dst)  
        # change to shutil.move if you want to move
        shutil.move(src, dst)  # uncomment this line and comment the above line to move instead of copy
        copied += 1

print(f"Moved {copied} files into split dirs under {NPZ_ROOT}.")
if missing:
    print(f"Missing {len(missing)} slides without matching NPZ (first 10): {missing[:10]}")


Copied 240 files into split dirs under C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_5x_norm2\unspecified.


In [5]:
def build_split_from_csv(npz_split_dir: Path, csv_path: Path, label_map):
    """Load slide vectors & labels for one split using a CSV mapping."""
    df = pd.read_csv(csv_path, dtype=str)
    assert {"slide_id","label"}.issubset(df.columns), f"CSV must have slide_id,label: {csv_path}"
    df["slide_id"] = df["slide_id"].str.strip()
    df["label"]    = df["label"].str.strip().map(label_map)

    X, y, ids = [], [], []
    missing_npz, missing_label = [], []

    # Build a quick index of available NPZ files in the split
    npz_index = {p.stem: p for p in npz_split_dir.glob("*.npz")}

    # Iterate labels; require matching npz
    for sid, lab in df[["slide_id","label"]].itertuples(index=False):
        p = npz_index.get(sid)
        if p is None:
            missing_npz.append(sid)
            continue
        qq, coords, mask = load_npz(p)
        X.append(slide_vector_from_qq(qq, mask, renorm=RENORM_ROWS))
        y.append(int(lab))
        ids.append(sid)

    # Any NPZ without a label?
    labeled_set = set(df["slide_id"])
    for sid in npz_index.keys():
        if sid not in labeled_set:
            missing_label.append(sid)

    if missing_npz:
        print(f"[WARN] {len(missing_npz)} slides in CSV missing NPZ in {npz_split_dir.name}: {missing_npz[:5]}{' ...' if len(missing_npz)>5 else ''}")
    if missing_label:
        print(f"[WARN] {len(missing_label)} NPZ files lack labels in CSV ({npz_split_dir.name}): {missing_label[:5]}{' ...' if len(missing_label)>5 else ''}")

    if not X:
        return np.empty((0,0), np.float32), np.array([], dtype=int), []
    X = np.vstack(X).astype(np.float32)
    y = np.asarray(y, dtype=int)
    return X, y, ids

In [6]:
train_dir = NPZ_ROOT / "train"
val_dir   = NPZ_ROOT / "val"
test_dir  = NPZ_ROOT / "test"

X_tr, y_tr, id_tr = build_split_from_csv(train_dir, CSV_TRAIN, LABEL_MAP)
X_va, y_va, id_va = build_split_from_csv(val_dir,   CSV_VAL,   LABEL_MAP)
X_te, y_te, id_te = build_split_from_csv(test_dir,  CSV_TEST,  LABEL_MAP)

print("train:", X_tr.shape, "val:", X_va.shape, "test:", X_te.shape)
assert (X_tr.shape[1] == 0) or (X_va.shape[1] in (0, X_tr.shape[1]))
assert (X_tr.shape[1] == 0) or (X_te.shape[1] in (0, X_tr.shape[1]))


train: (164, 16) val: (43, 16) test: (33, 16)


loading npz if already split into train, val, test directories

In [6]:
import os
from pathlib import Path
import re
import numpy as np
import pandas as pd

# === CONFIG ===
# Root folder that already contains split subfolders with NPZs
# e.g., NPZ_ROOT/train/*.npz, NPZ_ROOT/val/*.npz, NPZ_ROOT/test/*.npz
# NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_10x_norm2_filt\unspecified")

# CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
# CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
# CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
# CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")
# commented out above lines to avoid redefining variables for fusing magnifications

LABEL_MAP = {"FA": 0, "PT": 1}   # weak labels from CSV
RENORM_ROWS = True               # renormalize each patch prob row before averaging

# === UTILS ===
def canon(s: str) -> str:
    """Canonicalize ids/stems for tolerant matching (case/space/underscore/hyphen)."""
    return re.sub(r"[\s_-]+", "", str(s).strip().upper())

def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=False)
    qq = d["qq"]          # (N, K): patch x prototype probabilities
    coords = d["coords"]  # (N, 2)
    mask = d["mask"]      # (N,)
    return qq, coords, mask

def slide_vector_from_qq(qq, mask=None, renorm=True):
    """Aggregate patch probs → one K-dim feature per slide (mean qq)."""
    if mask is None:
        mask = np.ones((qq.shape[0],), dtype=bool)
    qq = qq[mask]
    if renorm:
        row_sums = qq.sum(axis=1, keepdims=True)
        qq = qq / (row_sums + 1e-12)
    return qq.mean(axis=0).astype(np.float32)  # shape (K,)

def index_npz_split_dir(npz_split_dir: Path):
    """
    Build a dict mapping canonical stem -> Path for quick lookup inside ONE split dir.
    If multiple files collide canonically, the first seen wins.
    """
    idx = {}
    for p in npz_split_dir.glob("*.npz"):
        idx.setdefault(canon(p.stem), p)
    return idx

def build_split_from_csv(npz_split_dir: Path, csv_path: Path, label_map: dict, renorm_rows: bool = True):
    """
    For a given split: read CSV, find each slide's .npz in that SAME split directory,
    load qq -> K-dim slide feature, and collect labels/ids.
    """
    if not csv_path.exists():
        # Allow missing splits (e.g., no test)
        return np.empty((0, 0), dtype=np.float32), np.array([], dtype=int), []

    df = pd.read_csv(csv_path, dtype=str)
    if not {"slide_id", "label"}.issubset(df.columns):
        raise ValueError(f"CSV must have columns slide_id,label: {csv_path}")

    df["slide_id"] = df["slide_id"].astype(str).str.strip()
    df["label"]    = df["label"].astype(str).str.strip().map(label_map)

    idx = index_npz_split_dir(npz_split_dir)
    X, y, ids = [], [], []
    missing_npz, missing_label = [], []

    for sid, lab in df[["slide_id", "label"]].itertuples(index=False):
        if pd.isna(lab):
            missing_label.append(sid)
            continue
        p = idx.get(canon(sid))
        if p is None:
            missing_npz.append(sid)
            continue
        qq, coords, mask = load_npz(p)
        X.append(slide_vector_from_qq(qq, mask, renorm=renorm_rows))
        y.append(int(lab))
        ids.append(sid)

    if missing_npz:
        print(f"[WARN] {len(missing_npz)} slides in CSV missing NPZ in '{npz_split_dir.name}': "
              f"{missing_npz[:5]}{' ...' if len(missing_npz) > 5 else ''}")
    if missing_label:
        print(f"[WARN] {len(missing_label)} slides have no valid label in CSV '{csv_path.name}': "
              f"{missing_label[:5]}{' ...' if len(missing_label) > 5 else ''}")

    if not X:
        return np.empty((0, 0), dtype=np.float32), np.array([], dtype=int), []

    X = np.vstack(X).astype(np.float32)
    y = np.asarray(y, dtype=int)
    return X, y, ids

# === BUILD ALL SPLITS (no moving/copying) ===
train_dir = NPZ_ROOT / "train"
val_dir   = NPZ_ROOT / "val"
test_dir  = NPZ_ROOT / "test"   # if you use 'split' instead of 'test', change this line

X_tr, y_tr, id_tr = build_split_from_csv(train_dir, CSV_TRAIN, LABEL_MAP, renorm_rows=RENORM_ROWS)
X_va, y_va, id_va = build_split_from_csv(val_dir,   CSV_VAL,   LABEL_MAP, renorm_rows=RENORM_ROWS)
X_te, y_te, id_te = build_split_from_csv(test_dir,  CSV_TEST,  LABEL_MAP, renorm_rows=RENORM_ROWS)

print("train:", X_tr.shape, "val:", X_va.shape, "test:", X_te.shape)
if X_tr.size and X_va.size:
    assert X_va.shape[1] == X_tr.shape[1], "Val K dimension != Train K dimension"
if X_tr.size and X_te.size:
    assert X_te.shape[1] == X_tr.shape[1], "Test K dimension != Train K dimension"
# commented out above lines to avoid redefining variables for fusing magnifications

train: (164, 16) val: (43, 16) test: (33, 16)


linear classifier, averaging patch probs per slide?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

clf = LogisticRegression(max_iter=2000, solver="liblinear", class_weight="balanced", random_state=0)
clf.fit(X_tr, y_tr)

def eval_split(name, X, y):
    if X.size == 0:
        print(f"{name}: (empty)")
        return
    prob = clf.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    acc = accuracy_score(y, pred)
    try:
        auc = roc_auc_score(y, prob)
    except ValueError:
        auc = float("nan")
    cm = confusion_matrix(y, pred)
    print(f"\n{name} — acc: {acc:.3f} | auc: {auc:.3f}")
    print("confusion_matrix (rows=true [FA=0, PT=1], cols=pred):\n", cm)
    print(classification_report(y, pred, target_names=["FA","PT"], digits=3))

eval_split("TRAIN", X_tr, y_tr)
eval_split("VAL",   X_va, y_va)
eval_split("TEST",  X_te, y_te)


In [ ]:
coef = clf.coef_.ravel()
K = coef.shape[0]
rank = np.argsort(coef)

print("\nTop + prototypes for PT:")
for k in rank[::-1][:5]:
    print(f"P{k}: {coef[k]:+.4f}")
print("\nTop − prototypes for FA:")
for k in rank[:5]:
    print(f"P{k}: {coef[k]:+.4f}")


MLP?

In [9]:
from pathlib import Path

# Root with split subfolders containing per-slide NPZs
NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\unspecified")  # contains train/, val/, test/

# CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")

LABEL_MAP = {"FA": 0, "PT": 1}

RENORM_ROWS = True   # renormalize each patch row before summarizing


In [10]:
import numpy as np
import pandas as pd

def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=False)
    return d["qq"], d["coords"], d["mask"]  # qq: (N,K)

def build_npz_index(split_dir: Path):
    """Map slide_id -> path for quick lookup."""
    return {p.stem: p for p in split_dir.glob("*.npz")}

def build_split_from_csv(npz_split_dir: Path, csv_path: Path, label_map):
    df = pd.read_csv(csv_path, dtype=str)
    assert {"slide_id","label"}.issubset(df.columns)
    df["slide_id"] = df["slide_id"].str.strip()
    df["label"]    = df["label"].str.strip().map(label_map)

    npz_idx = build_npz_index(npz_split_dir)
    rows = []
    missing_npz = []
    for sid, lab in df[["slide_id","label"]].itertuples(index=False):
        p = npz_idx.get(sid)
        if p is None:
            missing_npz.append(sid); continue
        rows.append((sid, int(lab), p))

    if missing_npz:
        print(f"[WARN] {len(missing_npz)} slides in {npz_split_dir.name} CSV missing NPZ. Examples:", missing_npz[:5])
    return rows  # list of (slide_id, label, path)


In [11]:
def per_proto_features(qq, mask=None, renorm=True, thresh=0.5, bins=(0.0,0.05,0.1,0.2,0.4,0.6,0.8,1.0000001)):
    """
    qq: (N,K) patch-by-prototype probabilities
    returns: feature vector of shape K * (5 + (len(bins)-1))
    """
    if mask is None:
        mask = np.ones((qq.shape[0],), dtype=bool)
    q = qq[mask]  # (Nm,K)
    if renorm:
        q = q / (q.sum(axis=1, keepdims=True) + 1e-12)

    N, K = q.shape
    feats = []
    for k in range(K):
        pk = q[:, k]
        stats = np.array([
            pk.mean(),
            pk.std(ddof=0),
            pk.max(),
            np.percentile(pk, 90),
            (pk > thresh).mean(),
        ], dtype=np.float32)
        h, _ = np.histogram(pk, bins=bins)
        h = (h / h.sum()).astype(np.float32)
        feats.append(np.concatenate([stats, h], axis=0))
    return np.concatenate(feats, axis=0).astype(np.float32)


In [12]:
def build_Xy(npz_split_dir: Path, csv_path: Path):
    rows = build_split_from_csv(npz_split_dir, csv_path, LABEL_MAP)
    X, y, ids = [], [], []
    for sid, lab, p in rows:
        qq, coords, mask = load_npz(p)
        x = per_proto_features(qq, mask, renorm=RENORM_ROWS)
        X.append(x); y.append(lab); ids.append(sid)
    if not X:
        return np.empty((0,0), np.float32), np.array([], dtype=int), []
    return np.vstack(X), np.asarray(y, dtype=int), ids

train_dir = NPZ_ROOT / "train"
val_dir   = NPZ_ROOT / "val"
test_dir  = NPZ_ROOT / "test"

X_tr, y_tr, id_tr = build_Xy(train_dir, CSV_TRAIN)
X_va, y_va, id_va = build_Xy(val_dir,   CSV_VAL)
X_te, y_te, id_te = build_Xy(test_dir,  CSV_TEST)

X_tr.shape, X_va.shape, X_te.shape


((164, 192), (43, 192), (33, 192))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,           # L2
        learning_rate_init=1e-3,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=0,
        verbose=False,
    )
)

mlp.fit(X_tr, y_tr)

def eval_split(name, X, y):
    if X.size == 0:
        print(f"{name}: (empty)"); return
    prob = mlp.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    acc  = accuracy_score(y, pred)
    try:
        auc = roc_auc_score(y, prob)
    except ValueError:
        auc = float("nan")
    cm   = confusion_matrix(y, pred)
    print(f"\n{name} — acc: {acc:.3f} | auc: {auc:.3f}")
    print("confusion_matrix (rows=true [FA=0, PT=1], cols=pred):\n", cm)
    print(classification_report(y, pred, target_names=["FA","PT"], digits=3))

eval_split("TRAIN", X_tr, y_tr)
eval_split("VAL",   X_va, y_va)
eval_split("TEST",  X_te, y_te)


train classifier on qq + majority voting

In [ ]:
# from pathlib import Path

# # Root with split subfolders containing per-slide NPZs
# NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\unspecified")  # contains train/, val/, test/

# # CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
# CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
# CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
# CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")

# LABEL_MAP = {"FA": 0, "PT": 1}

# RENORM_ROWS = True   # renormalize each patch row before summarizing


In [8]:
import numpy as np, pandas as pd

def load_npz(p: Path):
    d = np.load(p, allow_pickle=False)
    return d["qq"], d["coords"], d["mask"]  # qq: (N,K)

def build_npz_index(split_dir: Path):
    return {p.stem: p for p in split_dir.glob("*.npz")}

def rows_from_csv(npz_split_dir: Path, csv_path: Path):
    idx = build_npz_index(npz_split_dir)
    df = pd.read_csv(csv_path, dtype=str)
    df["slide_id"] = df["slide_id"].str.strip()
    df["label"]    = df["label"].str.strip().map(LABEL_MAP)
    rows, missing = [], []
    for sid, lab in df[["slide_id","label"]].itertuples(index=False):
        p = idx.get(sid)
        if p is None: missing.append(sid); continue
        rows.append((sid, int(lab), p))
    if missing: print(f"[WARN] {len(missing)} missing NPZ in {npz_split_dir.name}, e.g. {missing[:5]}")
    return rows  # [(slide_id, y, path), ...]


In [9]:
def build_patches(rows):
    X, y, slide_ids, patch_idx = [], [], [], []
    for sid, lab, p in rows:
        qq, _, mask = load_npz(p)              # (N,K)
        q = qq[mask]
        if RENORM_ROWS:
            q = q / (q.sum(axis=1, keepdims=True) + 1e-12)
        # features per patch = qq row (K-dim). Optionally add extras (entropy, max)
        X.append(q.astype(np.float32))
        y.append(np.full(q.shape[0], lab, dtype=int))
        slide_ids += [sid]*q.shape[0]
        patch_idx += list(range(q.shape[0]))
    if not X:
        return np.empty((0,0), np.float32), np.array([], int), [], []
    X = np.vstack(X); y = np.concatenate(y)
    return X, y, slide_ids, patch_idx

train_rows = rows_from_csv(NPZ_ROOT/"train", CSV_TRAIN)
val_rows   = rows_from_csv(NPZ_ROOT/"val",   CSV_VAL)
test_rows  = rows_from_csv(NPZ_ROOT/"test",  CSV_TEST)

X_tr, y_tr, sid_tr, pid_tr = build_patches(train_rows)
X_va, y_va, sid_va, pid_va = build_patches(val_rows)
X_te, y_te, sid_te, pid_te = build_patches(test_rows)

X_tr.shape, X_va.shape, X_te.shape


((419950, 16), (116150, 16), (81392, 16))

In [8]:
from sklearn.linear_model import LogisticRegression
from collections import Counter

# per-slide patch counts for weights
counts_tr = Counter(sid_tr)
w_tr = np.array([1.0 / counts_tr[s] for s in sid_tr], dtype=np.float32)

clf = LogisticRegression(max_iter=2000, solver="liblinear",
                         class_weight="balanced", random_state=0)
clf.fit(X_tr, y_tr, sample_weight=w_tr)


LogisticRegression(class_weight='balanced', max_iter=2000, random_state=0,
                   solver='liblinear')

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, balanced_accuracy_score

import numpy as np
# ---- PATCH-LEVEL METRICS ----
def patch_metrics(model, X, y, tau=0.5, name="PATCH"):
    p = model.predict_proba(X)[:, 1]
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    try:
        auc = roc_auc_score(y, p)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(y, yhat)
    rep = classification_report(y, yhat, target_names=["FA","PT"], digits=3)
    bal = balanced_accuracy_score(y, yhat) if len(np.unique(y))==2 else float("nan")
    print(f"\n{name} — acc: {acc:.3f} | auc: {auc:.3f} | bAcc: {bal:.3f}")
    print(cm, "\n", rep)
    return p, yhat, {"acc":acc, "auc":auc, "bacc":bal, "cm":cm, "report":rep}

# optional: see distribution of per-slide patch accuracy
def per_slide_patch_accuracy(sids, y, yhat):
    df = pd.DataFrame({"slide": sids, "correct": (y == yhat).astype(int)})
    acc_by_slide = df.groupby("slide")["correct"].mean().sort_values(ascending=False)
    return acc_by_slide  # fraction of correctly predicted patches per slide

def agg_metrics(X, y_slide_by_patch, slide_ids_by_patch, tau=0.5, mode="majority"):
    """
    Slide-level metrics.

    mode="soft":
        - slide score = mean probability over patches (soft)
        - ACC: 1 if soft >= tau
        - AUC: on soft

    mode="majority":
        - slide score = vote fraction = fraction of patches with prob >= tau
        - ACC: 1 if vote fraction >= 0.5
        - AUC: on vote fraction
    """
    p = clf.predict_proba(X)[:, 1]
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": p, "y": y_slide_by_patch})
    grouped = df.groupby("slide", sort=False)

    # Soft (mean prob) and vote-fraction (hard patch votes) per slide
    soft      = grouped["prob"].mean().to_numpy()
    voteFrac  = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
    slide_y   = grouped["y"].first().to_numpy()

    # --- extra metrics for auc calc---
    soft_auc  = roc_auc_score(slide_y, soft) if len(np.unique(slide_y)) == 2 else float("nan")
    vote_auc  = roc_auc_score(slide_y, voteFrac) if len(np.unique(slide_y)) == 2 else float("nan")
    # --------

    if mode == "majority":
        slide_pred = (voteFrac >= 0.5).astype(int)   # accuracy rule
        auc_scores = voteFrac                        # AUC on vote fraction
    elif mode == "soft":
        slide_pred = (soft >= tau).astype(int)       # accuracy rule
        auc_scores = soft                            # AUC on mean prob
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    acc = accuracy_score(slide_y, slide_pred)
    try:
        auc = roc_auc_score(slide_y, auc_scores)
    except ValueError:
        auc = float("nan")

    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    return acc, auc, cm, rep

# Tune tau on VAL (maximize balanced accuracy with soft vote; you can switch to majority)
def tune_tau_on_val(mode="majority"):
    ts = np.linspace(0,1,201)
    best, best_tau = -1, 0.5
    from sklearn.metrics import balanced_accuracy_score
    for t in ts:
        acc, auc, cm, rep = agg_metrics(X_va, y_va, sid_va, tau=t, mode=mode)
        # recompute balanced acc from cm
        tn, fp, fn, tp = cm.ravel()
        tpr = tp/(tp+fn+1e-9); tnr = tn/(tn+fp+1e-9)
        bal = 0.5*(tpr+tnr)
        if bal > best:
            best, best_tau = bal, t
    return best_tau, best

tau, bal = tune_tau_on_val(mode="majority")
print(f"Chosen τ={tau:.3f} (val balanced acc={bal:.3f})")

print("\nVAL (majority vote):")
acc, auc, cm, rep = agg_metrics(X_va, y_va, sid_va, tau=tau, mode="majority")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print("\nTEST (majority vote):")
acc, auc, cm, rep = agg_metrics(X_te, y_te, sid_te, tau=tau, mode="majority")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print("\nTEST (soft vote):")
acc, auc, cm, rep = agg_metrics(X_te, y_te, sid_te, tau=tau, mode="soft")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print('\nTRAIN (patch-level):')
p_tr, yhat_tr, metrics_tr = patch_metrics(clf, X_tr, y_tr, tau=0.5, name="TRAIN")
print('\nVAL (patch-level):')
p_va, yhat_va, metrics_va = patch_metrics(clf, X_va, y_va, tau=0.5, name="VAL")
print('\nTEST (patch-level):')
p_te, yhat_te, metrics_te = patch_metrics(clf, X_te, y_te, tau=0.5, name="TEST")

# print per slide patch accuracy distribution on TEST
acc_by_slide = per_slide_patch_accuracy(sid_te, y_te, yhat_te)
print("\nPer-slide patch accuracy (TEST):")
print(acc_by_slide.describe())
print(acc_by_slide)  # fraction of correctly predicted patches per slide


C:\Users\Vivian\AppData\Local\Temp\ipykernel_47040\1216064741.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac  = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in lab

Chosen τ=0.480 (val balanced acc=0.790)

VAL (majority vote):
acc=0.791 | auc=0.762
 [[16  5]
 [ 4 18]] 
               precision    recall  f1-score   support

          FA      0.800     0.762     0.780        21
          PT      0.783     0.818     0.800        22

    accuracy                          0.791        43
   macro avg      0.791     0.790     0.790        43
weighted avg      0.791     0.791     0.790        43


TEST (majority vote):
acc=0.455 | auc=0.835
 [[ 7 18]
 [ 0  8]] 
               precision    recall  f1-score   support

          FA      1.000     0.280     0.438        25
          PT      0.308     1.000     0.471         8

    accuracy                          0.455        33
   macro avg      0.654     0.640     0.454        33
weighted avg      0.832     0.455     0.446        33


TEST (soft vote):
acc=0.455 | auc=0.820
 [[ 7 18]
 [ 0  8]] 
               precision    recall  f1-score   support

          FA      1.000     0.280     0.438        25
 

: 

In [14]:
def agg_metrics(X, y_slide_by_patch, slide_ids_by_patch, tau=0.5, mode="majority", return_per_slide=False):
    """
    Slide-level metrics + optional per-slide breakdown.
    """
    p = clf.predict_proba(X)[:, 1]
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": p, "y": y_slide_by_patch})
    grouped = df.groupby("slide", sort=False)

    soft      = grouped["prob"].mean().to_numpy()
    voteFrac  = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
    slide_y   = grouped["y"].first().to_numpy()
    slide_ids = grouped["y"].apply(lambda g: g.name).to_numpy()

    if mode == "majority":
        slide_pred = (voteFrac >= 0.5).astype(int)
        auc_scores = voteFrac
    elif mode == "soft":
        slide_pred = (soft >= tau).astype(int)
        auc_scores = soft
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    acc = accuracy_score(slide_y, slide_pred)
    try:
        auc = roc_auc_score(slide_y, auc_scores)
    except ValueError:
        auc = float("nan")

    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)

    if return_per_slide:
        # Build DataFrame of per-slide correctness
        per_slide = pd.DataFrame({
            "slide_id": slide_ids,
            "true_label": slide_y,
            "pred_label": slide_pred,
            "correct": (slide_y == slide_pred).astype(int)
        })
        return acc, auc, cm, rep, per_slide
    else:
        return acc, auc, cm, rep


In [18]:
print("\nTEST (majority vote):")
acc, auc, cm, rep, per_slide = agg_metrics(X_te, y_te, sid_te, tau=tau, mode="majority", return_per_slide=True)
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

# -------------------------
# # Patch-level metrics grouped by slide
# df_patch = pd.DataFrame({"slide": sid_te, "correct_patch": (y_te == yhat_te).astype(int)})
# patch_acc_by_slide = df_patch.groupby("slide")["correct_patch"].mean()

# # Merge them
# # per_slide["slide_level_correct"] = (per_slide["true_label"] == per_slide["pred_label"]).astype(int)
# per_slide = per_slide.merge(patch_acc_by_slide, left_on="slide_id", right_index=True)
# per_slide = per_slide.rename(columns={"correct_patch": "patch_level_accuracy"})

# # Sort so you see best patch-level accuracy first
# print(per_slide.sort_values("patch_level_accuracy", ascending=False))
# # -------------------------

# # Sort slides by correctness
print("\nSlide-level accuracy (TEST) in descending order:")
print(per_slide.sort_values("correct", ascending=False))



TEST (majority vote):
acc=0.455 | auc=0.835
 [[ 7 18]
 [ 0  8]] 
               precision    recall  f1-score   support

          FA      1.000     0.280     0.438        25
          PT      0.308     1.000     0.471         8

    accuracy                          0.455        33
   macro avg      0.654     0.640     0.454        33
weighted avg      0.832     0.455     0.446        33


Slide-level accuracy (TEST) in descending order:
    slide_id  true_label  pred_label  correct
0     FA 56B           0           0        1
13   FA 70 B           0           0        1
31   PT 42 B           1           1        1
30   PT 41 B           1           1        1
29   PT 40 B           1           1        1
28   PT 39 B           1           1        1
27   PT 37 B           1           1        1
26   PT 36 B           1           1        1
25   PT 35 B           1           1        1
24   FA 86 B           0           0        1
20   FA 76 B           0           0        1
19  

C:\Users\Vivian\AppData\Local\Temp\ipykernel_47040\555743018.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac  = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()


MLP, training on each qq

In [ ]:
# ==== CONFIG ====
from pathlib import Path

# # Root with split subfolders containing per-slide NPZs
# NPZ_ROOT = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2\unspecified")  # contains train/, val/, test/

# # CSVs: each must have columns: slide_id,label   where label is "FA" or "PT"
# CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
# CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
# CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")

LABEL_MAP   = {"FA":0, "PT":1}
RENORM_ROWS = True          # renormalize each patch row so sums ≈ 1
CONF_THR    = None          # e.g., 0.6 to keep only confident patches (max qq >= 0.6); None = keep all

HIDDEN_SIZES = (128, 64)    # MLP hidden layers
LR_INIT      = 1e-3
ALPHA_L2     = 1e-4
MAX_EPOCHS   = 500
EARLY_STOP   = True

# ==== IMPORTS ====
# ----- RESAMPLE TO BALANCE PER-SLIDE + CLASS WITHOUT sample_weight -----
import numpy as np
from collections import defaultdict, Counter
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, balanced_accuracy_score

# Inspect sklearn version (optional)
import sklearn
print("scikit-learn version:", sklearn.__version__)

rng = np.random.RandomState(0)

def resample_patches(X, y, sid, per_slide_cap=1000, balance_classes=True, upsample_with_replacement=True, random_state=0):
    """
    - Caps patches per slide to avoid slides with huge N dominating.
    - Optionally balances the two classes by upsampling the minority.
    Returns X_res, y_res, sid_res (same shapes as X_res).
    """
    rng = np.random.RandomState(random_state)

    # 1) cap per slide
    slide_to_idxs = defaultdict(list)
    for i, s in enumerate(sid):
        slide_to_idxs[s].append(i)

    kept_idxs = []
    for s, idxs in slide_to_idxs.items():
        idxs = np.asarray(idxs)
        if len(idxs) > per_slide_cap:
            idxs = rng.choice(idxs, size=per_slide_cap, replace=False)
        kept_idxs.append(idxs)
    kept_idxs = np.concatenate(kept_idxs, axis=0)

    X_cap = X[kept_idxs]
    y_cap = y[kept_idxs]
    sid_cap = [sid[i] for i in kept_idxs]

    if not balance_classes:
        return X_cap, y_cap, sid_cap

    # 2) balance classes by upsampling minority (at patch level)
    idx_FA = np.where(y_cap == 0)[0]
    idx_PT = np.where(y_cap == 1)[0]
    n_FA, n_PT = len(idx_FA), len(idx_PT)

    if n_FA == 0 or n_PT == 0:
        # nothing to balance
        return X_cap, y_cap, sid_cap

    if n_FA > n_PT:
        need = n_FA - n_PT
        add = rng.choice(idx_PT, size=need, replace=upsample_with_replacement)
        idx_bal = np.concatenate([np.arange(len(y_cap)), add])
    elif n_PT > n_FA:
        need = n_PT - n_FA
        add = rng.choice(idx_FA, size=need, replace=upsample_with_replacement)
        idx_bal = np.concatenate([np.arange(len(y_cap)), add])
    else:
        idx_bal = np.arange(len(y_cap))

    X_bal = X_cap[idx_bal]
    y_bal = y_cap[idx_bal]
    sid_bal = [sid_cap[i] for i in idx_bal]
    return X_bal, y_bal, sid_bal

# --- apply resampling on your already-built patch sets ---
# (uses variables X_tr, y_tr, sid_tr from your previous cells)
X_tr_bal, y_tr_bal, sid_tr_bal = resample_patches(
    X_tr, y_tr, sid_tr,
    per_slide_cap=1000,          # tune: 500, 1000, 2000
    balance_classes=True,
    upsample_with_replacement=True,
    random_state=0
)
print("after resample — train:", X_tr_bal.shape, "class counts:", Counter(y_tr_bal))

# ----- MLP pipeline (no sample_weight) -----
pipe = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        learning_rate_init=1e-3,
        alpha=1e-4,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=0,
        verbose=False,
    ))
])

pipe.fit(X_tr_bal, y_tr_bal)

# ----- Slide-level aggregation helpers (majority / soft vote) -----
import pandas as pd

def patch_metrics(model, X, y, tau=0.5, name="PATCH"):
    """
    Patch-level metrics (no slide aggregation).
    """
    p = model.predict_proba(X)[:, 1]
    yhat = (p >= tau).astype(int)

    acc = accuracy_score(y, yhat)
    try:
        auc = roc_auc_score(y, p)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(y, yhat)
    rep = classification_report(y, yhat, target_names=["FA","PT"], digits=3)
    bal = balanced_accuracy_score(y, yhat) if len(np.unique(y))==2 else float("nan")

    print(f"\n{name} — acc: {acc:.3f} | auc: {auc:.3f} | bAcc: {bal:.3f}")
    print(cm, "\n", rep)
    return p, yhat, {"acc":acc, "auc":auc, "bacc":bal, "cm":cm, "report":rep}

def agg_metrics(*args, tau=0.5, mode="majority"):
    """
    Slide-level metrics with flexible calling convention.

    Usage A (global clf):     agg_metrics(X, y_slide, slide_ids, tau=..., mode=...)
    Usage B (pass model):     agg_metrics(model, X, y_slide, slide_ids, tau=..., mode=...)

    mode="soft":
        - slide score = mean probability over patches (soft)
        - ACC: 1 if soft >= tau
        - AUC: on soft

    mode="majority":
        - slide score = vote fraction = fraction of patches with prob >= tau
        - ACC: 1 if vote fraction >= 0.5
        - AUC: on vote fraction
    """
    # Parse arguments (support both call styles)
    if len(args) == 4 and hasattr(args[0], "predict_proba"):
        model, X, y_slide_by_patch, slide_ids_by_patch = args
    elif len(args) == 3:
        # Use global clf
        model = globals().get("clf", None)
        if model is None or not hasattr(model, "predict_proba"):
            raise ValueError("No model provided and global `clf` not found or lacks predict_proba.")
        X, y_slide_by_patch, slide_ids_by_patch = args
    else:
        raise TypeError("agg_metrics expects (X,y,sids,...) or (model,X,y,sids,...)")

    # Patch-level probabilities
    p = model.predict_proba(X)[:, 1]

    # Build per-slide table
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": p, "y": y_slide_by_patch})
    grouped = df.groupby("slide", sort=False)

    # Aggregates per slide
    soft = grouped["prob"].mean().to_numpy()                            # mean prob
    voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()  # vote fraction
    slide_y = grouped["y"].first().to_numpy()

    # Accuracy rule + AUC score source
    if mode == "majority":
        slide_pred = (voteFrac >= 0.5).astype(int)  # accuracy
        auc_scores = voteFrac                       # AUC aligns with majority
    elif mode == "soft":
        slide_pred = (soft >= tau).astype(int)      # accuracy
        auc_scores = soft                           # AUC aligns with soft
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    # Metrics
    acc = accuracy_score(slide_y, slide_pred)
    try:
        auc = roc_auc_score(slide_y, auc_scores)
    except ValueError:
        auc = float("nan")

    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    return acc, auc, cm, rep

# ----- tune threshold on VAL (majority vote; you can switch to 'soft') -----
ts = np.linspace(0, 1, 201)
best_tau, best_bal = 0.5, -1
for t in ts:
    acc, auc, cm, rep = agg_metrics(pipe, X_va, y_va, sid_va, tau=t, mode="majority")
    
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
        bal = 0.5 * ((tp/(tp+fn+1e-9)) + (tn/(tn+fp+1e-9)))
    else:
        bal = 0.0
    if bal > best_bal:
        best_bal, best_tau = bal, t
print(f"Chosen τ on VAL: {best_tau:.3f} (balanced acc={best_bal:.3f})")

# ----- evaluate -----
print("\nVAL — majority:")
acc, auc, cm, rep = agg_metrics(pipe, X_va, y_va, sid_va, tau=best_tau, mode="majority")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print("\nTEST — majority:")
acc, auc, cm, rep = agg_metrics(pipe, X_te, y_te, sid_te, tau=best_tau, mode="majority")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print("\nTEST — soft (same τ):")
acc, auc, cm, rep = agg_metrics(pipe, X_te, y_te, sid_te, tau=best_tau, mode="soft")
print(f"acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print('===== PATCH-LEVEL METRICS =====')
print('\nTRAIN (patch-level):')
p_tr, yhat_tr, metrics_tr = patch_metrics(pipe, X_tr, y_tr, tau=0.5, name="TRAIN")

print('\nVAL (patch-level):')
p_va, yhat_va, metrics_va = patch_metrics(pipe, X_va, y_va, tau=0.5, name="VAL")

print('\nTEST (patch-level):')
p_te, yhat_te, metrics_te = patch_metrics(pipe, X_te, y_te, tau=0.5, name="TEST")


pytorch MLP

In [10]:
# ==== PyTorch MLP Patch-Level Classifier ====
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, balanced_accuracy_score
import pandas as pd

# ---- ClassifierHead definition (as you provided) ----
class ClassifierHead(nn.Module):
    def __init__(self, input_dim, num_classes, MLP='1L'):
        super().__init__()
        self.MLP = MLP
        if MLP == '1L':
            self.fc = nn.Linear(input_dim, num_classes)
        elif MLP == '3L':
            self.net = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Linear(128, num_classes)
            )
        elif MLP == '5L':
            self.net = nn.Sequential(
                nn.Linear(input_dim, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Linear(64, num_classes)
            )

    def forward(self, x):
        if self.MLP == '1L':
            return self.fc(x)
        else:
            return self.net(x)

# ---- Config ----
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
WD = 1e-4
MLP_TYPE = '3L'  # choose '1L', '3L', or '5L'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data prep (convert numpy -> torch tensors) ----
def make_loader(X, y, batch_size, shuffle=True):
    X_tensor = torch.from_numpy(X).float()
    y_tensor = torch.from_numpy(y).long()
    ds = TensorDataset(X_tensor, y_tensor)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_tr, y_tr, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_va, y_va, BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_te, y_te, BATCH_SIZE, shuffle=False)

# ---- Model, optimizer, loss ----
model = ClassifierHead(input_dim=X_tr.shape[1], num_classes=2, MLP=MLP_TYPE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

# ---- Training loop ----
best_val_auc, best_state = 0.0, None
for epoch in range(EPOCHS):
    # train
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    avg_loss = total_loss / len(train_loader.dataset)

    # validate
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_probs.append(probs); all_y.append(yb.numpy())
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)
    try:
        val_auc = roc_auc_score(all_y, all_probs)
    except ValueError:
        val_auc = 0.0
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = model.state_dict()
    print(f"Epoch {epoch+1}/{EPOCHS} | Train loss {avg_loss:.4f} | Val AUC {val_auc:.3f}")

# ---- Load best model ----
model.load_state_dict(best_state)

# ---- Evaluation helper (patch-level) ----
def eval_loader(loader, name="TEST"):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            all_probs.append(probs); all_y.append(yb.numpy())
    all_probs = np.concatenate(all_probs)
    all_y = np.concatenate(all_y)
    yhat = (all_probs >= 0.5).astype(int)
    acc = accuracy_score(all_y, yhat)
    auc = roc_auc_score(all_y, all_probs) if len(np.unique(all_y))==2 else float("nan")
    bal = balanced_accuracy_score(all_y, yhat) if len(np.unique(all_y))==2 else float("nan")
    cm  = confusion_matrix(all_y, yhat)
    rep = classification_report(all_y, yhat, target_names=["FA","PT"], digits=3)
    print(f"\n{name} — acc={acc:.3f} | auc={auc:.3f} | bAcc={bal:.3f}")
    print(cm, "\n", rep)
    return all_probs, all_y

print("===== PATCH-LEVEL METRICS =====")
p_tr, y_tr_true = eval_loader(train_loader, "TRAIN")
p_va, y_va_true = eval_loader(val_loader,   "VAL")
p_te, y_te_true = eval_loader(test_loader,  "TEST")

# ---- Slide-level aggregation (majority vs soft) ----
def agg_slide_metrics(probs, y_by_patch, slide_ids_by_patch, tau=0.5, mode="majority"):
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": probs, "y": y_by_patch})
    grouped = df.groupby("slide", sort=False)
    soft = grouped["prob"].mean().to_numpy()
    voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
    slide_y = grouped["y"].first().to_numpy()
    if mode=="majority":
        slide_pred = (voteFrac >= 0.5).astype(int)
        auc_scores = voteFrac
    else:
        slide_pred = (soft >= tau).astype(int)
        auc_scores = soft
    acc = accuracy_score(slide_y, slide_pred)
    auc = roc_auc_score(slide_y, auc_scores) if len(np.unique(slide_y))==2 else float("nan")
    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    print(f"\n{name} — acc={acc:.3f} | auc={auc:.3f}\n", cm, "\n", rep)

print("===== SLIDE-LEVEL METRICS =====")
agg_slide_metrics(p_va, y_va, sid_va, tau=0.5, mode="majority")
agg_slide_metrics(p_te, y_te, sid_te, tau=0.5, mode="majority")
agg_slide_metrics(p_te, y_te, sid_te, tau=0.5, mode="soft")


Epoch 1/50 | Train loss 0.5858 | Val AUC 0.642
Epoch 2/50 | Train loss 0.5715 | Val AUC 0.639
Epoch 3/50 | Train loss 0.5645 | Val AUC 0.640
Epoch 4/50 | Train loss 0.5601 | Val AUC 0.640
Epoch 5/50 | Train loss 0.5561 | Val AUC 0.639
Epoch 6/50 | Train loss 0.5532 | Val AUC 0.646
Epoch 7/50 | Train loss 0.5502 | Val AUC 0.651
Epoch 8/50 | Train loss 0.5480 | Val AUC 0.651
Epoch 9/50 | Train loss 0.5460 | Val AUC 0.650
Epoch 10/50 | Train loss 0.5443 | Val AUC 0.648
Epoch 11/50 | Train loss 0.5423 | Val AUC 0.649
Epoch 12/50 | Train loss 0.5402 | Val AUC 0.647
Epoch 13/50 | Train loss 0.5385 | Val AUC 0.637
Epoch 14/50 | Train loss 0.5373 | Val AUC 0.639
Epoch 15/50 | Train loss 0.5362 | Val AUC 0.639
Epoch 16/50 | Train loss 0.5353 | Val AUC 0.632
Epoch 17/50 | Train loss 0.5344 | Val AUC 0.649
Epoch 18/50 | Train loss 0.5330 | Val AUC 0.643
Epoch 19/50 | Train loss 0.5320 | Val AUC 0.637
Epoch 20/50 | Train loss 0.5310 | Val AUC 0.639
Epoch 21/50 | Train loss 0.5303 | Val AUC 0.636
E

C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\2614054614.py:145: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()


NameError: name 'name' is not defined

In [11]:
def agg_slide_metrics(probs, y_by_patch, slide_ids_by_patch, tau=0.5, mode="majority", split_name="SPLIT"):
    """
    probs: np.ndarray [N]  -> PT probability per patch
    y_by_patch: np.ndarray [N] -> label per patch (slide label repeated per patch)
    slide_ids_by_patch: array-like [N] -> slide id for each patch
    tau: decision threshold
    mode: "majority" or "soft"
    split_name: string to show in logs (e.g., "VAL" or "TEST")
    """
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": probs, "y": y_by_patch})
    grouped = df.groupby("slide", sort=False)

    # Per-slide aggregation
    soft     = grouped["prob"].mean().to_numpy()
    voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
    slide_y  = grouped["y"].first().to_numpy()

    if mode == "majority":
        slide_pred = (voteFrac >= 0.5).astype(int)
        auc_scores = voteFrac
    elif mode == "soft":
        slide_pred = (soft >= tau).astype(int)
        auc_scores = soft
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    acc = accuracy_score(slide_y, slide_pred)
    try:
        auc = roc_auc_score(slide_y, auc_scores)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    print(f"\n{split_name} — SLIDE [{mode}] τ={tau:.2f} | acc={acc:.3f} | auc={auc:.3f}")
    print(cm, "\n", rep)
    return {"acc":acc, "auc":auc, "cm":cm, "report":rep}


print("===== SLIDE-LEVEL METRICS =====")
agg_slide_metrics(p_va, y_va, sid_va, tau=0.5, mode="majority", split_name="VAL")
agg_slide_metrics(p_te, y_te, sid_te, tau=0.5, mode="majority", split_name="TEST")
agg_slide_metrics(p_te, y_te, sid_te, tau=0.5, mode="soft",     split_name="TEST")


===== SLIDE-LEVEL METRICS =====

VAL — SLIDE [majority] τ=0.50 | acc=0.744 | auc=0.766
[[14  7]
 [ 4 18]] 
               precision    recall  f1-score   support

          FA      0.778     0.667     0.718        21
          PT      0.720     0.818     0.766        22

    accuracy                          0.744        43
   macro avg      0.749     0.742     0.742        43
weighted avg      0.748     0.744     0.743        43


TEST — SLIDE [majority] τ=0.50 | acc=0.364 | auc=0.495
[[ 7 18]
 [ 3  5]] 
               precision    recall  f1-score   support

          FA      0.700     0.280     0.400        25
          PT      0.217     0.625     0.323         8

    accuracy                          0.364        33
   macro avg      0.459     0.453     0.361        33
weighted avg      0.583     0.364     0.381        33


TEST — SLIDE [soft] τ=0.50 | acc=0.394 | auc=0.540
[[ 5 20]
 [ 0  8]] 
               precision    recall  f1-score   support

          FA      1.000     0.200

C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\3760484793.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\3760484793.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\i

{'acc': 0.3939393939393939,
 'auc': 0.54,
 'cm': array([[ 5, 20],
        [ 0,  8]], dtype=int64),
 'report': '              precision    recall  f1-score   support\n\n          FA      1.000     0.200     0.333        25\n          PT      0.286     1.000     0.444         8\n\n    accuracy                          0.394        33\n   macro avg      0.643     0.600     0.389        33\nweighted avg      0.827     0.394     0.360        33\n'}

In [12]:
# ============================== CONFIG ==============================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix, balanced_accuracy_score)
import pandas as pd
from collections import defaultdict, Counter
import random

# ---- User knobs ----
MLP_TYPE      = '3L'     # '1L' (linear), '3L', or '5L'
BATCH_SIZE    = 256
EPOCHS        = 100
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 15       # early-stop patience on val AUC

# Imbalance handling (train-time)
USE_RESAMPLE          = True    # cap patches per slide + upsample minority (preprocessing)
PER_SLIDE_CAP         = 1000
UPSAMPLE_WITH_REPL    = True

USE_WEIGHTED_SAMPLER  = True    # DataLoader samples inversely to class frequency
USE_CLASS_WEIGHTS     = True    # CrossEntropyLoss(weight=[w_FA,w_PT]) using TRAIN frequencies

SEED = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== SEEDING ==============================
def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# ====================== ASSERT INPUT ARRAYS EXIST ====================
def must_have(name, obj):
    assert obj is not None, f"`{name}` is None. Provide {name} before running."
def must_match_len(name_a, a, name_b, b):
    assert len(a) == len(b), f"Length mismatch: {name_a}={len(a)} vs {name_b}={len(b)}"

for nm in ["X_tr","y_tr","sid_tr","X_va","y_va","sid_va","X_te","y_te","sid_te"]:
    must_have(nm, globals().get(nm, None))

must_match_len("X_tr", X_tr, "y_tr", y_tr); must_match_len("X_tr", X_tr, "sid_tr", sid_tr)
must_match_len("X_va", X_va, "y_va", y_va); must_match_len("X_va", X_va, "sid_va", sid_va)
must_match_len("X_te", X_te, "y_te", y_te); must_match_len("X_te", X_te, "sid_te", sid_te)

# ======================= OPTIONAL RESAMPLING =========================
from collections import defaultdict

def resample_patches(X, y, sid, per_slide_cap=1000, balance_classes=True,
                     upsample_with_replacement=True, random_state=0):
    """
    - Caps patches per slide to avoid slides with huge N dominating.
    - Optionally balances classes at patch level by upsampling the minority.
    """
    rng = np.random.RandomState(random_state)

    slide_to_idxs = defaultdict(list)
    for i, s in enumerate(sid):
        slide_to_idxs[s].append(i)

    kept_idxs = []
    for s, idxs in slide_to_idxs.items():
        idxs = np.asarray(idxs)
        if len(idxs) > per_slide_cap:
            idxs = rng.choice(idxs, size=per_slide_cap, replace=False)
        kept_idxs.append(idxs)
    kept_idxs = np.concatenate(kept_idxs, axis=0)

    X_cap = X[kept_idxs]
    y_cap = y[kept_idxs]
    sid_cap = [sid[i] for i in kept_idxs]

    if not balance_classes:
        return X_cap, y_cap, sid_cap

    idx_FA = np.where(y_cap == 0)[0]
    idx_PT = np.where(y_cap == 1)[0]
    n_FA, n_PT = len(idx_FA), len(idx_PT)

    if n_FA == 0 or n_PT == 0:
        return X_cap, y_cap, sid_cap

    if n_FA > n_PT:
        need = n_FA - n_PT
        add = rng.choice(idx_PT, size=need, replace=upsample_with_replacement)
        idx_bal = np.concatenate([np.arange(len(y_cap)), add])
    elif n_PT > n_FA:
        need = n_PT - n_FA
        add = rng.choice(idx_FA, size=need, replace=upsample_with_replacement)
        idx_bal = np.concatenate([np.arange(len(y_cap)), add])
    else:
        idx_bal = np.arange(len(y_cap))

    X_bal = X_cap[idx_bal]
    y_bal = y_cap[idx_bal]
    sid_bal = [sid_cap[i] for i in idx_bal]
    return X_bal, y_bal, sid_bal

if USE_RESAMPLE:
    X_tr, y_tr, sid_tr = resample_patches(
        X_tr, y_tr, sid_tr,
        per_slide_cap=PER_SLIDE_CAP,
        balance_classes=True,
        upsample_with_replacement=UPSAMPLE_WITH_REPL,
        random_state=SEED
    )
    print("After resample — train:", X_tr.shape, "class counts:", Counter(y_tr))

# =========================== DATASETS ================================
X_tr_tensor = torch.from_numpy(X_tr).float()
y_tr_tensor = torch.from_numpy(y_tr).long()
train_ds = TensorDataset(X_tr_tensor, y_tr_tensor)

X_va_tensor = torch.from_numpy(X_va).float()
y_va_tensor = torch.from_numpy(y_va).long()
val_ds = TensorDataset(X_va_tensor, y_va_tensor)

X_te_tensor = torch.from_numpy(X_te).float()
y_te_tensor = torch.from_numpy(y_te).long()
test_ds = TensorDataset(X_te_tensor, y_te_tensor)

# ===================== WEIGHTED SAMPLER (optional) ===================
if USE_WEIGHTED_SAMPLER:
    # inverse-frequency weights per sample (patch)
    cls_counts = np.bincount(y_tr.astype(int), minlength=2)
    inv_freq = 1.0 / (cls_counts + 1e-9)
    sample_weights = inv_freq[y_tr.astype(int)]
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(y_tr),  # one epoch ~ len(y_tr) draws
        replacement=True
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=False)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)

val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# =========================== MODEL HEAD ==============================
class ClassifierHead(nn.Module):
    def __init__(self, input_dim, num_classes, MLP='1L'):
        super().__init__()
        self.MLP = MLP
        if MLP == '1L':
            self.fc = nn.Linear(input_dim, num_classes)
        elif MLP == '3L':
            self.net = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Linear(128, num_classes)
            )
        elif MLP == '5L':
            self.net = nn.Sequential(
                nn.Linear(input_dim, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Linear(64, num_classes)
            )
        else:
            raise ValueError("MLP must be one of: '1L', '3L', '5L'")

    def forward(self, x):
        if self.MLP == '1L':
            return self.fc(x)
        else:
            return self.net(x)

model = ClassifierHead(input_dim=X_tr.shape[1], num_classes=2, MLP=MLP_TYPE).to(device)

# ==================== CLASS-WEIGHTED LOSS (optional) =================
if USE_CLASS_WEIGHTS:
    n_FA = (y_tr == 0).sum()
    n_PT = (y_tr == 1).sum()
    # normalize to sum ~1 for stability (any proportional scaling works)
    w_FA = 0.5 / (n_FA + 1e-9)
    w_PT = 0.5 / (n_PT + 1e-9)
    class_weights = torch.tensor([w_FA, w_PT], dtype=torch.float, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"class weights: FA={float(w_FA):.6f}, PT={float(w_PT):.6f}")
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ========================== TRAIN / VALIDATE =========================
def evaluate_probs(loader):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs); all_y.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

best_val_auc, best_state, epochs_no_improve = -1.0, None, 0
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    tr_loss = total_loss / len(train_loader.dataset)

    # Validation AUC for early stopping
    p_va_probs, y_va_true = evaluate_probs(val_loader)
    try:
        val_auc = roc_auc_score(y_va_true, p_va_probs)
    except ValueError:
        val_auc = 0.0

    improved = val_auc > best_val_auc + 1e-4
    if improved:
        best_val_auc = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:03d} | train loss {tr_loss:.4f} | val AUC {val_auc:.3f} "
          f"{'[*]' if improved else ''}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping (no improve {PATIENCE} epochs). Best val AUC = {best_val_auc:.3f}")
        break

# Load best weights
if best_state is not None:
    model.load_state_dict(best_state)

# ========================== PATCH-LEVEL METRICS ======================
def patch_level_report(probs, true, name="SPLIT"):
    yhat = (probs >= 0.5).astype(int)
    acc = accuracy_score(true, yhat)
    auc = roc_auc_score(true, probs) if len(np.unique(true))==2 else float("nan")
    bal = balanced_accuracy_score(true, yhat) if len(np.unique(true))==2 else float("nan")
    cm  = confusion_matrix(true, yhat)
    rep = classification_report(true, yhat, target_names=["FA","PT"], digits=3)
    print(f"\n{name} — PATCH | acc={acc:.3f} | auc={auc:.3f} | bAcc={bal:.3f}")
    print(cm, "\n", rep)

p_tr_probs, y_tr_true = evaluate_probs(DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False))
p_va_probs, y_va_true = evaluate_probs(val_loader)
p_te_probs, y_te_true = evaluate_probs(test_loader)

print("\n===== PATCH-LEVEL METRICS =====")
patch_level_report(p_tr_probs, y_tr_true, "TRAIN")
patch_level_report(p_va_probs, y_va_true, "VAL")
patch_level_report(p_te_probs, y_te_true, "TEST")

# ========================= SLIDE-LEVEL HELPERS =======================
def agg_slide_metrics(probs, y_by_patch, slide_ids_by_patch, tau=0.5, mode="majority", split_name="SPLIT"):
    """
    probs: np.ndarray [N]  (PT probability per patch)
    y_by_patch: np.ndarray [N] (label per patch; usually slide-label repeated per patch)
    slide_ids_by_patch: array-like [N] (slide id for each patch)
    """
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": probs, "y": y_by_patch})
    grouped = df.groupby("slide", sort=False)

    soft     = grouped["prob"].mean().to_numpy()                         # mean prob per slide
    voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()  # majority vote fraction
    slide_y  = grouped["y"].first().to_numpy()

    if mode == "majority":
        slide_pred = (voteFrac >= 0.5).astype(int)
        auc_scores = voteFrac
    elif mode == "soft":
        slide_pred = (soft >= tau).astype(int)
        auc_scores = soft
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    acc = accuracy_score(slide_y, slide_pred)
    try:
        auc = roc_auc_score(slide_y, auc_scores)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    print(f"\n{split_name} — SLIDE [{mode}] τ={tau:.2f} | acc={acc:.3f} | auc={auc:.3f}")
    print(cm, "\n", rep)
    return {"acc":acc, "auc":auc, "cm":cm, "report":rep}

def tune_tau_on_val(val_probs, val_y_by_patch, val_sid, mode="majority"):
    taus = np.linspace(0.0, 1.0, 201)
    best_tau, best_bal = 0.5, -1
    df = pd.DataFrame({"slide": val_sid, "prob": val_probs, "y": val_y_by_patch})
    g  = df.groupby("slide", sort=False)
    for t in taus:
        soft     = g["prob"].mean().to_numpy()
        voteFrac = g.apply(lambda r: (r["prob"] >= t).mean()).to_numpy()
        slide_y  = g["y"].first().to_numpy()

        if mode == "majority":
            pred = (voteFrac >= 0.5).astype(int)
        else:
            pred = (soft >= t).astype(int)

        cm = confusion_matrix(slide_y, pred)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            bal = 0.5 * ((tp/(tp+fn+1e-9)) + (tn/(tn+fp+1e-9)))
        else:
            bal = 0.0
        if bal > best_bal:
            best_bal, best_tau = bal, t
    print(f"Chosen τ on VAL ({mode}): {best_tau:.3f} (balanced acc={best_bal:.3f})")
    return best_tau

# ========================== SLIDE-LEVEL EVAL =========================
print("\n===== SLIDE-LEVEL METRICS =====")
val_tau = tune_tau_on_val(p_va_probs, y_va, sid_va, mode="majority")
agg_slide_metrics(p_va_probs, y_va, sid_va, tau=val_tau, mode="majority", split_name="VAL")
agg_slide_metrics(p_te_probs, y_te, sid_te, tau=val_tau, mode="majority", split_name="TEST")
agg_slide_metrics(p_te_probs, y_te, sid_te, tau=val_tau, mode="soft",     split_name="TEST")


After resample — train: (175544, 16) class counts: Counter({0: 87772, 1: 87772})
class weights: FA=0.000006, PT=0.000006
Epoch 001 | train loss 0.6040 | val AUC 0.644 [*]
Epoch 002 | train loss 0.5943 | val AUC 0.643 
Epoch 003 | train loss 0.5893 | val AUC 0.636 
Epoch 004 | train loss 0.5849 | val AUC 0.640 
Epoch 005 | train loss 0.5809 | val AUC 0.628 
Epoch 006 | train loss 0.5768 | val AUC 0.637 
Epoch 007 | train loss 0.5747 | val AUC 0.632 
Epoch 008 | train loss 0.5730 | val AUC 0.642 
Epoch 009 | train loss 0.5690 | val AUC 0.628 
Epoch 010 | train loss 0.5684 | val AUC 0.641 
Epoch 011 | train loss 0.5655 | val AUC 0.645 [*]
Epoch 012 | train loss 0.5658 | val AUC 0.630 
Epoch 013 | train loss 0.5610 | val AUC 0.624 
Epoch 014 | train loss 0.5601 | val AUC 0.637 
Epoch 015 | train loss 0.5595 | val AUC 0.638 
Epoch 016 | train loss 0.5579 | val AUC 0.640 
Epoch 017 | train loss 0.5585 | val AUC 0.640 
Epoch 018 | train loss 0.5559 | val AUC 0.632 
Epoch 019 | train loss 0.55

C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\2787376945.py:318: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = g.apply(lambda r: (r["prob"] >= t).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\2787376945.py:318: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = g.apply(lambda r: (r["prob"] >= t).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\

Chosen τ on VAL (majority): 0.430 (balanced acc=0.720)

VAL — SLIDE [majority] τ=0.43 | acc=0.721 | auc=0.764
[[14  7]
 [ 5 17]] 
               precision    recall  f1-score   support

          FA      0.737     0.667     0.700        21
          PT      0.708     0.773     0.739        22

    accuracy                          0.721        43
   macro avg      0.723     0.720     0.720        43
weighted avg      0.722     0.721     0.720        43


TEST — SLIDE [majority] τ=0.43 | acc=0.394 | auc=0.545
[[ 6 19]
 [ 1  7]] 
               precision    recall  f1-score   support

          FA      0.857     0.240     0.375        25
          PT      0.269     0.875     0.412         8

    accuracy                          0.394        33
   macro avg      0.563     0.557     0.393        33
weighted avg      0.715     0.394     0.384        33


TEST — SLIDE [soft] τ=0.43 | acc=0.394 | auc=0.580
[[ 5 20]
 [ 0  8]] 
               precision    recall  f1-score   support

          

C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\2787376945.py:318: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = g.apply(lambda r: (r["prob"] >= t).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\2787376945.py:318: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = g.apply(lambda r: (r["prob"] >= t).mean()).to_numpy()
C:\Users\Vivian\AppData\Local\Temp\ipykernel_9488\

{'acc': 0.3939393939393939,
 'auc': 0.5800000000000001,
 'cm': array([[ 5, 20],
        [ 0,  8]], dtype=int64),
 'report': '              precision    recall  f1-score   support\n\n          FA      1.000     0.200     0.333        25\n          PT      0.286     1.000     0.444         8\n\n    accuracy                          0.394        33\n   macro avg      0.643     0.600     0.389        33\nweighted avg      0.827     0.394     0.360        33\n'}

In [13]:
# ================= PATCH-LEVEL METRICS (after training) =================
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, roc_auc_score, balanced_accuracy_score,
                             confusion_matrix, classification_report, average_precision_score)

# If you already have these from earlier, you can reuse them.
# Otherwise, recreate simple DataLoaders from numpy arrays:
def make_loader_from_numpy(X, y, batch_size=256, shuffle=False):
    X_t = torch.from_numpy(X).float()
    y_t = torch.from_numpy(y).long()
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle, drop_last=False)

batch_size_eval = 512
train_loader_eval = make_loader_from_numpy(X_tr, y_tr, batch_size=batch_size_eval, shuffle=False)
val_loader_eval   = make_loader_from_numpy(X_va, y_va, batch_size=batch_size_eval, shuffle=False)
test_loader_eval  = make_loader_from_numpy(X_te, y_te, batch_size=batch_size_eval, shuffle=False)

def predict_patch_probs(model, loader, device):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()  # PT prob
            all_probs.append(probs)
            all_y.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)

def patch_metrics_report(probs, y_true, tau=0.5, name="PATCH"):
    """
    probs: ndarray [N] of PT probabilities for patches
    y_true: ndarray [N] of true patch labels (0=FA, 1=PT) — if you only have slide labels,
            these can be slide labels repeated per patch (that's fine for patch-level eval).
    tau: decision threshold for converting probs -> hard labels
    """
    y_pred = (probs >= tau).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred) if len(np.unique(y_true)) == 2 else float("nan")
    try:
        auc  = roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else float("nan")
        ap   = average_precision_score(y_true, probs) if len(np.unique(y_true)) == 2 else float("nan")
    except ValueError:
        auc, ap = float("nan"), float("nan")

    cm   = confusion_matrix(y_true, y_pred)
    rep  = classification_report(y_true, y_pred, target_names=["FA","PT"], digits=3)

    print(f"\n{name} — PATCH (τ={tau:.2f}) | acc={acc:.3f} | bAcc={bacc:.3f} | ROC-AUC={auc:.3f} | PR-AUC={ap:.3f}")
    print(cm, "\n", rep)
    return {
        "acc": acc, "bacc": bacc, "auc": auc, "ap": ap,
        "cm": cm, "report": rep
    }

# ---- Run patch-level evaluation for each split
p_tr_probs, y_tr_true = predict_patch_probs(model, train_loader_eval, device)
p_va_probs, y_va_true = predict_patch_probs(model, val_loader_eval,   device)
p_te_probs, y_te_true = predict_patch_probs(model, test_loader_eval,  device)

print("\n===== PATCH-LEVEL RESULTS =====")
patch_metrics_report(p_tr_probs, y_tr_true, tau=0.5, name="TRAIN")
patch_metrics_report(p_va_probs, y_va_true, tau=0.5, name="VAL")
patch_metrics_report(p_te_probs, y_te_true, tau=0.5, name="TEST")

# (Optional) If you tuned a different τ on validation for patch decisions, reuse it here:
# best_tau_patch = 0.45  # example
# patch_metrics_report(p_va_probs, y_va_true, tau=best_tau_patch, name="VAL@bestτ")
# patch_metrics_report(p_te_probs, y_te_true, tau=best_tau_patch, name="TEST@bestτ")



===== PATCH-LEVEL RESULTS =====

TRAIN — PATCH (τ=0.50) | acc=0.711 | bAcc=0.711 | ROC-AUC=0.789 | PR-AUC=0.806
[[66569 21203]
 [29520 58252]] 
               precision    recall  f1-score   support

          FA      0.693     0.758     0.724     87772
          PT      0.733     0.664     0.697     87772

    accuracy                          0.711    175544
   macro avg      0.713     0.711     0.710    175544
weighted avg      0.713     0.711     0.710    175544


VAL — PATCH (τ=0.50) | acc=0.600 | bAcc=0.605 | ROC-AUC=0.647 | PR-AUC=0.699
[[34959 18190]
 [28235 34766]] 
               precision    recall  f1-score   support

          FA      0.553     0.658     0.601     53149
          PT      0.657     0.552     0.600     63001

    accuracy                          0.600    116150
   macro avg      0.605     0.605     0.600    116150
weighted avg      0.609     0.600     0.600    116150


TEST — PATCH (τ=0.50) | acc=0.533 | bAcc=0.543 | ROC-AUC=0.574 | PR-AUC=0.319
[[33388 30

{'acc': 0.5326936308236682,
 'bacc': 0.5433623167743817,
 'auc': 0.5737732288843631,
 'ap': 0.31858985528829964,
 'cm': array([[33388, 30274],
        [ 7761,  9969]], dtype=int64),
 'report': '              precision    recall  f1-score   support\n\n          FA      0.811     0.524     0.637     63662\n          PT      0.248     0.562     0.344     17730\n\n    accuracy                          0.533     81392\n   macro avg      0.530     0.543     0.491     81392\nweighted avg      0.689     0.533     0.573     81392\n'}

attempting to fuse all magnfications

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
from collections import Counter

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, classification_report
)

# === MULTI-MAG CONFIG ===
MAG_ROOTS = {
    "10x":  Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_10x_norm2_filt\unspecified"),
    "5x":   Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_5x_norm2_filt\unspecified"),
    "2p5x": Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\patch_probs\qq_2.5x_norm2_filt\unspecified"),
}
CSV_TRAIN = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\train.csv")
CSV_VAL   = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\val.csv")
CSV_TEST  = Path(r"C:\Users\Vivian\Documents\PANTHER\PANTHER\src\splits\FA_PT_5x_norm2_k=0\test.csv")
LABEL_MAP = {"FA":0, "PT":1}
RENORM_ROWS = True

# ---- assumes you already defined elsewhere: load_npz, etc. If not, uncomment these helpers. ----
def canon(s: str) -> str:
    return re.sub(r"[\s_-]+", "", str(s).strip().upper())

def index_npz_split_dir(npz_split_dir: Path):
    idx = {}
    for p in npz_split_dir.glob("*.npz"):
        idx.setdefault(canon(p.stem), p)
    return idx

# === PATCH-LEVEL LOADER (NOT slide-mean) ===
def build_patch_dataset_from_csv(npz_split_dir: Path, csv_path: Path, label_map: dict,
                                 renorm_rows: bool = True):
    """
    Return patch-level X (all qq rows), y (slide label per patch), sid (slide_id per patch).
    """
    if not csv_path.exists():
        return (np.empty((0, 0), dtype=np.float32),
                np.array([], dtype=int),
                np.array([], dtype=object))

    df = pd.read_csv(csv_path, dtype=str)
    df["slide_id"] = df["slide_id"].astype(str).str.strip()
    df["label"]    = df["label"].astype(str).str.strip().map(label_map)

    idx = index_npz_split_dir(npz_split_dir)
    X_list, y_list, sid_list = [], [], []
    missing_npz, missing_label = [], []

    for sid, lab in df[["slide_id", "label"]].itertuples(index=False):
        if pd.isna(lab):
            missing_label.append(sid)
            continue
        p = idx.get(canon(sid))
        if p is None:
            missing_npz.append(sid)
            continue
        d = np.load(p, allow_pickle=False)
        qq = d["qq"]          # (N,K)
        mask = d["mask"]      # (N,)
        if mask is None:
            mask = np.ones((qq.shape[0],), dtype=bool)
        qq = qq[mask]
        if renorm_rows:
            row_sums = qq.sum(axis=1, keepdims=True)
            qq = qq / (row_sums + 1e-12)
        X_list.append(qq.astype(np.float32))
        y_list.append(np.full((qq.shape[0],), int(lab), dtype=int))
        sid_list.append(np.array([sid]*qq.shape[0], dtype=object))

    if missing_npz:
        print(f"[WARN] {len(missing_npz)} slides missing NPZ in '{npz_split_dir.name}': "
              f"{missing_npz[:5]}{' ...' if len(missing_npz)>5 else ''}")
    if missing_label:
        print(f"[WARN] {len(missing_label)} slides have no valid label in CSV '{csv_path.name}': "
              f"{missing_label[:5]}{' ...' if len(missing_label)>5 else ''}")

    if not X_list:
        return (np.empty((0, 0), dtype=np.float32),
                np.array([], dtype=int),
                np.array([], dtype=object))

    X   = np.vstack(X_list)
    y   = np.concatenate(y_list)
    sid = np.concatenate(sid_list)
    return X, y, sid

def load_patch_splits_for_root(root: Path):
    dtrain, dval, dtest = root/"train", root/"val", root/"test"
    X_tr, y_tr, sid_tr = build_patch_dataset_from_csv(dtrain, CSV_TRAIN, LABEL_MAP, renorm_rows=RENORM_ROWS)
    X_va, y_va, sid_va = build_patch_dataset_from_csv(dval,   CSV_VAL,   LABEL_MAP, renorm_rows=RENORM_ROWS)
    X_te, y_te, sid_te = build_patch_dataset_from_csv(dtest,  CSV_TEST,  LABEL_MAP, renorm_rows=RENORM_ROWS)
    return {"train": (X_tr, y_tr, sid_tr),
            "val":   (X_va, y_va, sid_va),
            "test":  (X_te, y_te, sid_te)}

# === YOUR METRIC HELPERS BUT WITH EXPLICIT clf PARAM ===
def agg_metrics_for_clf(clf, X, y_slide_by_patch, slide_ids_by_patch, tau=0.5, mode="majority"):
    p = clf.predict_proba(X)[:, 1]
    df = pd.DataFrame({"slide": slide_ids_by_patch, "prob": p, "y": y_slide_by_patch})
    grouped = df.groupby("slide", sort=False)

    soft     = grouped["prob"].mean().to_numpy()
    voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
    slide_y  = grouped["y"].first().to_numpy()

    if mode == "majority":
        slide_pred, auc_scores = (voteFrac >= 0.5).astype(int), voteFrac
    elif mode == "soft":
        slide_pred, auc_scores = (soft >= tau).astype(int), soft
    else:
        raise ValueError("mode must be 'majority' or 'soft'")

    acc = accuracy_score(slide_y, slide_pred)
    try: auc = roc_auc_score(slide_y, auc_scores)
    except ValueError: auc = float("nan")
    cm  = confusion_matrix(slide_y, slide_pred)
    rep = classification_report(slide_y, slide_pred, target_names=["FA","PT"], digits=3)
    return acc, auc, cm, rep, soft, voteFrac, slide_y

def tune_tau_on_val_for_clf(clf, X_va, y_va, sid_va, mode="majority"):
    ts = np.linspace(0,1,201)
    best, best_tau = -1, 0.5
    for t in ts:
        acc, auc, cm, rep, soft, vf, slide_y = agg_metrics_for_clf(clf, X_va, y_va, sid_va, tau=t, mode=mode)
        tn, fp, fn, tp = cm.ravel()
        tpr = tp/(tp+fn+1e-9); tnr = tn/(tn+fp+1e-9)
        bal = 0.5*(tpr+tnr)
        if bal > best:
            best, best_tau = bal, t
    return best_tau, best

# === TRAIN / EVAL PER MAG AT PATCH LEVEL (SAME AS YOUR ORIGINAL LOGIC) ===
def fit_patch_lr(X_tr, y_tr, sid_tr):
    counts_tr = Counter(sid_tr)  # per-slide patch counts
    w_tr = np.array([1.0 / counts_tr[s] for s in sid_tr], dtype=np.float32)
    clf = LogisticRegression(max_iter=2000, solver="liblinear",
                             class_weight="balanced", random_state=0)
    clf.fit(X_tr, y_tr, sample_weight=w_tr)
    return clf

# 1) Load per-mag PATCH-LEVEL data
data = {mag: load_patch_splits_for_root(root) for mag, root in MAG_ROOTS.items()}
for mag in data:
    X_tr, y_tr, sid_tr = data[mag]["train"]
    X_va, y_va, sid_va = data[mag]["val"]
    X_te, y_te, sid_te = data[mag]["test"]
    print(mag, "patch shapes:", X_tr.shape, X_va.shape, X_te.shape)

# 2) Train one PATCH-level classifier per mag + tune τ on VAL (same as before)
clfs, taus = {}, {}
permag_val_soft, permag_test_soft = {}, {}
y_val_slides_ref, y_test_slides_ref = None, None
val_frames, test_frames = [], []

for mag in MAG_ROOTS:
    X_tr, y_tr, sid_tr = data[mag]["train"]
    X_va, y_va, sid_va = data[mag]["val"]
    X_te, y_te, sid_te = data[mag]["test"]

    assert X_tr.size, f"No training data for {mag}"
    clf = fit_patch_lr(X_tr, y_tr, sid_tr)
    clfs[mag] = clf

    tau, bal = tune_tau_on_val_for_clf(clf, X_va, y_va, sid_va, mode="majority")
    taus[mag] = tau
    print(f"[{mag}] chosen τ={tau:.3f} (val best bAcc={bal:.3f})")

    # VAL slide-level soft scores (mean patch prob)
    acc, auc, cm, rep, soft_va, vf_va, y_va_slides = agg_metrics_for_clf(
        clf, X_va, y_va, sid_va, tau=tau, mode="soft")
    print(f"[{mag}] VAL: acc={acc:.3f} | auc={auc:.3f} | n_slides={len(y_va_slides)}")

    # TEST slide-level soft scores
    acc, auc, cm, rep, soft_te, vf_te, y_te_slides = agg_metrics_for_clf(
        clf, X_te, y_te, sid_te, tau=tau, mode="soft")
    print(f"[{mag}] TEST: acc={acc:.3f} | auc={auc:.3f} | n_slides={len(y_te_slides)}")

    # Build frames for alignment/fusion
    # Extract consistent slide order from the grouped objects again to pair scores to IDs
    df_va = pd.DataFrame({"slide": sid_va})
    slides_va_order = df_va.groupby("slide", sort=False).size().index.tolist()
    # y_va_slides already matches that order because of agg_metrics_for_clf implementation
    dv = pd.DataFrame({"slide_id": slides_va_order, "y": y_va_slides, f"p_{mag}": soft_va})

    df_te = pd.DataFrame({"slide": sid_te})
    slides_te_order = df_te.groupby("slide", sort=False).size().index.tolist()
    dt = pd.DataFrame({"slide_id": slides_te_order, "y": y_te_slides, f"p_{mag}": soft_te})

    val_frames.append(dv)
    test_frames.append(dt)

# 3) Align slides across mags & fuse (label-space)
from functools import reduce

def inner_join_on_slide(frames):
    merged = reduce(lambda l, r: pd.merge(l, r[["slide_id"] + [c for c in r.columns if c.startswith("p_")]],
                                          on="slide_id", how="inner"), frames)
    y_series = frames[0].set_index("slide_id").loc[merged["slide_id"], "y"].to_numpy()
    return merged, y_series

val_merged, y_val_slides = inner_join_on_slide(val_frames)
test_merged, y_test_slides = inner_join_on_slide(test_frames)
print("VAL merged columns:", list(val_merged.columns))
print("TEST merged columns:", list(test_merged.columns))

# === ATTENTION-BASED FUSION ON SLIDE-LEVEL PROBS ===
# === IMPROVED ATTENTION FUSION (logit inputs + calibration + entropy bonus) ===
import numpy as np, torch, torch.nn as nn, torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# ---- 1) Prepare slide-level inputs as LOGITS and standardize ----
def prob_to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

pcols = [c for c in val_merged.columns if c.startswith("p_")]
Xval_prob = val_merged[pcols].to_numpy().astype(np.float32)   # [N_val, 3]
Xte_prob  = test_merged[pcols].to_numpy().astype(np.float32)

X_val = prob_to_logit(Xval_prob).astype(np.float32)
X_te  = prob_to_logit(Xte_prob).astype(np.float32)

y_val = y_val_slides.astype(np.int64)
y_te  = y_test_slides.astype(np.int64)

scaler = StandardScaler()
X_val = scaler.fit_transform(X_val).astype(np.float32)
X_te  = scaler.transform(X_te).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_mags = X_val.shape[1]

# ---- helpers ----
def balanced_acc(y_true, scores, tau=0.5):
    yhat = (scores >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()
    tpr = tp / (tp + fn + 1e-9)
    tnr = tn / (tn + fp + 1e-9)
    return 0.5 * (tpr + tnr)

def tune_tau(y_true, scores):
    ts = np.linspace(0, 1, 201)
    best, best_t = -1, 0.5
    for t in ts:
        bal = balanced_acc(y_true, scores, tau=t)
        if bal > best:
            best, best_t = bal, t
    return best_t

# ---- 2) Internal split on VAL for early stopping ----
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
(train_idx, val_idx) = next(sss.split(X_val, y_val))
X_tr_i, y_tr_i = X_val[train_idx], y_val[train_idx]
X_va_i, y_va_i = X_val[val_idx],  y_val[val_idx]

class NpDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y.astype(np.float32))
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

BATCH = min(64, max(8, len(X_tr_i)//8))
dl_tr = torch.utils.data.DataLoader(NpDataset(X_tr_i, y_tr_i), batch_size=BATCH, shuffle=True)
dl_va = torch.utils.data.DataLoader(NpDataset(X_va_i, y_va_i), batch_size=256, shuffle=False)

# ---- 3) Attention model with temperature + dropout + entropy bonus ----
class MagAttention(nn.Module):
    def __init__(self, n_mags: int, temp: float = 2.0):
        super().__init__()
        self.temp = temp
        self.attn = nn.Sequential(
            nn.Linear(n_mags, 16),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(16, n_mags)
        )
        # Initialize last layer to near-zero so we start ~uniform
        nn.init.zeros_(self.attn[-1].weight)
        nn.init.zeros_(self.attn[-1].bias)
        self.head = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(inplace=True),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        # x: [B, n_mags] (standardized logits)
        logits_w = self.attn(x) / self.temp          # temperature to flatten
        w = torch.softmax(logits_w, dim=1)           # [B, n_mags]
        fused = (x * w).sum(dim=1, keepdim=True)     # weighted sum in logit-space
        out = self.head(fused).squeeze(1)            # logits
        return out, w

model = MagAttention(n_mags=n_mags, temp=2.0).to(device)

# Class imbalance (BCE with pos_weight)
pos_weight = torch.tensor([(y_tr_i == 0).sum() / ((y_tr_i == 1).sum() + 1e-9)],
                          dtype=torch.float32, device=device)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

opt = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-2)
EPOCHS, PATIENCE = 400, 40
lambda_ent = 0.02   # encourages diverse attention (higher entropy)

def entropy_w(w, eps=1e-9):
    # mean entropy across batch; maximize it -> add with +lambda_ent
    return - (w.clamp_min(eps) * (w.clamp_min(eps).log())).sum(dim=1).mean()

best_state, best_metric, patience = None, -np.inf, PATIENCE
for epoch in range(1, EPOCHS+1):
    model.train()
    tr_loss = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        logits, w = model(xb)
        loss = bce(logits, yb) - lambda_ent * entropy_w(w)  # maximize entropy
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        tr_loss += loss.item() * xb.size(0)
    tr_loss /= len(dl_tr.dataset)

    # Validate (balanced accuracy with tuned tau)
    model.eval()
    with torch.no_grad():
        logits_v, wv = model(torch.from_numpy(X_va_i).to(device))
        prob_v = torch.sigmoid(logits_v).cpu().numpy()
    tau_v = tune_tau(y_va_i, prob_v)
    bal_v = balanced_acc(y_va_i, prob_v, tau=tau_v)

    if bal_v > best_metric + 1e-5:
        best_metric = bal_v
        patience = PATIENCE
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience -= 1
        if patience <= 0:
            break

# ---- 4) Retrain on ALL VAL for epochs_best and evaluate on TEST ----
epochs_best = EPOCHS - patience
model.load_state_dict(best_state)
model2 = MagAttention(n_mags=n_mags, temp=2.0).to(device)
model2.load_state_dict(best_state)  # warm start
opt2 = optim.AdamW(model2.parameters(), lr=2e-3, weight_decay=5e-2)

dl_all = torch.utils.data.DataLoader(NpDataset(X_val, y_val), batch_size=BATCH, shuffle=True)
for _ in range(max(1, epochs_best)):
    for xb, yb in dl_all:
        xb, yb = xb.to(device), yb.to(device)
        logits, w = model2(xb)
        loss = bce(logits, yb) - lambda_ent * entropy_w(w)
        opt2.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), 5.0)
        opt2.step()

model2.eval()
with torch.no_grad():
    prob_val_full = torch.sigmoid(model2(torch.from_numpy(X_val).to(device))[0]).cpu().numpy()
    prob_te_full  = torch.sigmoid(model2(torch.from_numpy(X_te ).to(device))[0]).cpu().numpy()

tau_final = tune_tau(y_val, prob_val_full)

def report_split(name, y, prob, tau):
    yhat = (prob >= tau).astype(int)
    acc  = accuracy_score(y, yhat)
    try:
        auc = roc_auc_score(y, prob)
    except ValueError:
        auc = float("nan")
    print(f"[ATTN*] {name}: acc={acc:.3f} | auc={auc:.3f} | τ={tau:.2f}")
    print(classification_report(y, yhat, target_names=["FA","PT"], digits=3))

print(f"[ATTN*] Internal-VAL best bAcc={best_metric:.3f}, epochs_best={epochs_best}")
report_split("VAL(full)", y_val, prob_val_full, tau_final)
report_split("TEST",      y_te,  prob_te_full,  tau_final)

with torch.no_grad():
    _, W_te = model2(torch.from_numpy(X_te).to(device))
W_te = W_te.cpu().numpy()
print("\n[ATTN*] Mean attention weights per mag:")
for i, c in enumerate(pcols):
    print(f"  {c}: mean(w)={W_te[:, i].mean():.3f}")


# # === STACKING (meta LR on slide-level probs) ===
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
# import numpy as np

# # Columns holding per-mag slide probabilities (created above)
# pcols = [c for c in val_merged.columns if c.startswith("p_")]
# X_val_meta = val_merged[pcols].to_numpy()
# X_te_meta  = test_merged[pcols].to_numpy()

# y_val_meta = y_val_slides
# y_te_meta  = y_test_slides

# # Small, robust meta-learner
# meta = Pipeline([
#     ("scaler", StandardScaler(with_mean=True, with_std=True)),
#     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=0))
# ])
# meta.fit(X_val_meta, y_val_meta)

# # Meta probabilities
# val_meta_prob = meta.predict_proba(X_val_meta)[:, 1]
# te_meta_prob  = meta.predict_proba(X_te_meta)[:, 1]

# # Reuse your tune_tau helper if you already defined it; otherwise define it here:
# def _tune_tau(y, s):
#     ts = np.linspace(0, 1, 201)
#     best, best_t = -1, 0.5
#     for t in ts:
#         yhat = (s >= t).astype(int)
#         tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
#         tpr = tp / (tp + fn + 1e-9)
#         tnr = tn / (tn + fp + 1e-9)
#         bal = 0.5 * (tpr + tnr)
#         if bal > best:
#             best, best_t = bal, t
#     return best_t

# tau_meta = _tune_tau(y_val_meta, val_meta_prob)

# # Report
# def _report(y, s, tau):
#     yhat = (s >= tau).astype(int)
#     acc  = accuracy_score(y, yhat)
#     auc  = roc_auc_score(y, s)
#     print(f"acc={acc:.3f} | auc={auc:.3f} | τ={tau:.2f}")
#     print(classification_report(y, yhat, target_names=["FA","PT"], digits=3))

# print("[STACK] VAL");  _report(y_val_meta, val_meta_prob, tau_meta)
# print("[STACK] TEST"); _report(y_te_meta,  te_meta_prob,  tau_meta)

# # === SIMPLE FUSION (mean of probs) ===
# pcols = [c for c in val_merged.columns if c.startswith("p_")]
# val_mean  = val_merged[pcols].mean(axis=1).to_numpy()
# test_mean = test_merged[pcols].mean(axis=1).to_numpy()

# acc_val = accuracy_score(y_val_slides, (val_mean>=0.5).astype(int))
# auc_val = roc_auc_score(y_val_slides, val_mean)
# acc_te  = accuracy_score(y_test_slides, (test_mean>=0.5).astype(int))
# auc_te  = roc_auc_score(y_test_slides, test_mean)
# print(f"[FUSED mean] VAL acc={acc_val:.3f} | auc={auc_val:.3f}")
# print(f"[FUSED mean] TEST acc={acc_te:.3f} | auc={auc_te:.3f}")

# # (Optional) Logit-mean fusion + threshold tuning on VAL
# def prob_to_logit(p, eps=1e-6):
#     p = np.clip(p, eps, 1-eps)
#     return np.log(p/(1-p))
# def logit_to_prob(z):
#     return 1/(1+np.exp(-z))

# Z_val = np.stack([prob_to_logit(val_merged[c].to_numpy()) for c in pcols], axis=1)
# Z_te  = np.stack([prob_to_logit(test_merged[c].to_numpy()) for c in pcols], axis=1)
# val_logitmean  = logit_to_prob(Z_val.mean(axis=1))
# test_logitmean = logit_to_prob(Z_te.mean(axis=1))

# def tune_tau(y, s):
#     ts = np.linspace(0,1,201)
#     best, best_t = -1, 0.5
#     for t in ts:
#         yhat = (s >= t).astype(int)
#         tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
#         bal = 0.5*((tp/(tp+fn+1e-9)) + (tn/(tn+fp+1e-9)))
#         if bal > best:
#             best, best_t = bal, t
#     return best_t

# tau_fused = tune_tau(y_val_slides, val_logitmean)
# acc_val2 = accuracy_score(y_val_slides, (val_logitmean>=tau_fused).astype(int))
# auc_val2 = roc_auc_score(y_val_slides, val_logitmean)
# acc_te2  = accuracy_score(y_test_slides, (test_logitmean>=tau_fused).astype(int))
# auc_te2  = roc_auc_score(y_test_slides, test_logitmean)
# print(f"[FUSED logit-mean + τ={tau_fused:.2f}] VAL acc={acc_val2:.3f} | auc={auc_val2:.3f}")
# print(f"[FUSED logit-mean + τ={tau_fused:.2f}] TEST acc={acc_te2:.3f} | auc={auc_te2:.3f}")


10x patch shapes: (1044596, 16) (289282, 16) (141073, 16)
5x patch shapes: (419950, 16) (116150, 16) (55772, 16)
2p5x patch shapes: (108465, 16) (29914, 16) (14104, 16)


C:\Users\Vivian\AppData\Local\Temp\ipykernel_59612\329917727.py:107: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labe

[10x] chosen τ=0.480 (val best bAcc=0.790)
[10x] VAL: acc=0.791 | auc=0.764 | n_slides=43
[10x] TEST: acc=0.455 | auc=0.820 | n_slides=33


C:\Users\Vivian\AppData\Local\Temp\ipykernel_59612\329917727.py:107: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labe

[5x] chosen τ=0.485 (val best bAcc=0.767)
[5x] VAL: acc=0.791 | auc=0.768 | n_slides=43
[5x] TEST: acc=0.576 | auc=0.825 | n_slides=33


C:\Users\Vivian\AppData\Local\Temp\ipykernel_59612\329917727.py:107: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  voteFrac = grouped.apply(lambda g: (g["prob"] >= tau).mean()).to_numpy()
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\Vivian\anaconda3\envs\panther\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labe

[2p5x] chosen τ=0.490 (val best bAcc=0.766)
[2p5x] VAL: acc=0.744 | auc=0.751 | n_slides=43
[2p5x] TEST: acc=0.636 | auc=0.830 | n_slides=33
VAL merged columns: ['slide_id', 'y', 'p_10x', 'p_5x', 'p_2p5x']
TEST merged columns: ['slide_id', 'y', 'p_10x', 'p_5x', 'p_2p5x']
[ATTN*] Internal-VAL best bAcc=0.775, epochs_best=400
[ATTN*] VAL(full): acc=0.791 | auc=0.764 | τ=0.53
              precision    recall  f1-score   support

          FA      0.800     0.762     0.780        21
          PT      0.783     0.818     0.800        22

    accuracy                          0.791        43
   macro avg      0.791     0.790     0.790        43
weighted avg      0.791     0.791     0.790        43

[ATTN*] TEST: acc=0.545 | auc=0.825 | τ=0.53
              precision    recall  f1-score   support

          FA      1.000     0.400     0.571        25
          PT      0.348     1.000     0.516         8

    accuracy                          0.545        33
   macro avg      0.674     0.700 